# RDD: the resilient distributed dataset

_Apache Spark_

**Spark's original abstraction — an immutable, partitioned collection you transform lazily and process in parallel across a cluster.**

When data is too big for one machine, you spread it across many machines and process the pieces
       in parallel. Spark's first answer to "how do I hold and compute on that spread-out data" was the
       RDD (Resilient Distributed Dataset). It is the foundation everything else in Spark is built on.

       Read the name one word at a time:

---

This notebook is a practice scaffold for the **RDD: the resilient distributed dataset** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PySpark

In [ ]:
# Colab: !pip install pyspark   (the notebook's setup cell installs it)
from pyspark.sql import SparkSession

# local[*] = run Spark on this one Colab VM, using all its cores as "workers".
spark = SparkSession.builder.master("local[*]").appName("rdd-demo").getOrCreate()
sc = spark.sparkContext          # the SparkContext: the entry point for RDDs

# ---- Build an RDD (Resilient Distributed Dataset) from a Python list ----
# The opening of "A Tale of Two Cities" (public domain), one line per record.
lines = sc.parallelize([
    "It was the best of times it was the worst of times",
    "it was the age of wisdom it was the age of foolishness",
    "it was the epoch of belief it was the epoch of incredulity",
    "it was the season of Light it was the season of Darkness",
    "it was the spring of hope it was the winter of despair",
], numSlices=4)                  # split into 4 partitions = the unit of parallelism

print("partitions:", lines.getNumPartitions())   # -> 4

# ---- TRANSFORMATIONS (all LAZY: nothing runs here, just a recipe) ----
counts = (
    lines
      .flatMap(lambda line: line.lower().split())   # each line -> its words (flattened)
      .map(lambda w: (w, 1))                          # each word -> (word, 1)
      .reduceByKey(lambda a, b: a + b)                # add the 1's sharing a word
)
print("nothing has run yet -- counts is just a recipe:", type(counts).__name__)

# ---- ACTIONS (trigger the whole chain to actually execute) ----
top = counts.takeOrdered(5, key=lambda kv: -kv[1])   # action: top 5 by count
print("top 5:", top)            # [('it',10),('was',10),('the',10),('of',10),('times',2)]
print("distinct words:", counts.count())             # action: 20
print("first 3 (any order):", counts.take(3))        # action: cheap peek
# print(counts.collect())       # action: ALL pairs to driver -- fine here (tiny),
                                 # but NEVER on a huge RDD: it OOMs the driver.

# ---- LINEAGE: the recorded chain Spark replays to rebuild a lost partition ----
print(counts.toDebugString().decode())  # shows ShuffledRDD <- MapPartitionsRDD <- ... <- ParallelCollectionRDD

spark.stop()

# NOTE: for STRUCTURED data, prefer a DataFrame -- it gets the Catalyst optimizer
# (RDDs do not), so the same word count runs faster:
#   from pyspark.sql.functions import explode, split, lower, col
#   df = spark.createDataFrame([(l,) for l in lines.collect()], ["line"])
#   (df.select(explode(split(lower(col("line")), " ")).alias("w"))
#      .groupBy("w").count().orderBy(col("count").desc()).show(5))

## Visualize the data & results

_What does the classic RDD word count (flatMap -> map -> reduceByKey) actually return? Here are the top word counts a Spark reduceByKey would produce on a small real text, computed reproducibly with a numpy/pandas word count._

In [ ]:
import re
import pandas as pd

# The famous 60-word opening of "A Tale of Two Cities" (public domain text).
text = '''It was the best of times, it was the worst of times,
it was the age of wisdom, it was the age of foolishness,
it was the epoch of belief, it was the epoch of incredulity,
it was the season of Light, it was the season of Darkness,
it was the spring of hope, it was the winter of despair.'''

# Same logic an RDD does (flatMap split -> map (w,1) -> reduceByKey add),
# expressed as a reproducible pandas word count:
words = re.findall(r'[a-z]+', text.lower())   # flatMap: lines -> words
counts = pd.Series(words).value_counts()      # map+reduceByKey: tally per word

print('total words :', len(words))            # -> 60
print('distinct    :', counts.size)           # -> 20
print(counts.head(8))
# it       10
# was      10
# the      10
# of       10
# times     2
# age       2
# epoch     2
# season    2

top = counts.head(8)
print('labels:', list(top.index))             # ['it','was','the','of','times','age','epoch','season']
print('values:', list(top.values))            # [10, 10, 10, 10, 2, 2, 2, 2]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You run counts = lines.flatMap(lambda l: l.split()).map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b) and the cell returns instantly with no output. Did Spark count the words? What is the next line you write to actually get answers?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List which calls are transformations and which is an action. — _flatMap, map, and reduceByKey are all transformations &mdash; they are lazy and only record a recipe._
- Realize the chain built a new RDD but ran nothing, because no action was called. — _Spark executes only when an action demands a concrete result; until then there is nothing to do._
- Add an action, e.g. counts.take(5) or counts.collect() (if small) or counts.count(). — _An action triggers the whole lineage to run and materializes results._

**Answer:** No &mdash; Spark counted nothing yet. flatMap, map, and reduceByKey are lazy transformations; the cell just built a new RDD and recorded the recipe, which is why it returned instantly. To force the work, call an action: counts.take(5) to peek at a few (word, count) pairs, counts.count() for the number of distinct words, or counts.collect() only if the result is small enough for the driver. The action is where the computation (and any hidden error) actually happens.

</details>

**Problem 2.** A teammate's job sums values per key with pairs.groupByKey().mapValues(lambda vs: sum(vs)) and it is painfully slow and occasionally crashes one worker. What is going wrong, and what one-line change fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize groupByKey is a wide transformation that shuffles every raw value across the network. — _It moves all values for a key to one machine before combining, so a hot key floods a single worker._
- Note the goal is just a per-key sum, which is associative. — _An associative combine can be done partially on each partition first, then merged &mdash; no need to ship raw values._
- Replace with pairs.reduceByKey(lambda a, b: a + b). — _reduceByKey combines values locally on each partition first, so only small partial sums cross the network._

**Answer:** groupByKey is the problem: it is a wide transformation that shuffles every raw value over the network and piles all of a key's values onto one machine before combining &mdash; slow, and a recipe for an out-of-memory crash on a skewed key. Since the operation is just an associative sum, switch to pairs.reduceByKey(lambda a, b: a + b). reduceByKey does a local combine on each partition first (a "map-side combine"), so only one small partial sum per key per partition is shuffled. Same answer, far less data moved.

</details>

**Problem 3.** Midway through a long Spark job, a worker machine dies and takes several partitions with it. The job keeps running and still produces the correct result &mdash; no data was replicated anywhere. How is that possible, and what is the one requirement that makes it safe?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that each RDD records its lineage &mdash; the full chain of transformations from durable input. — _The lineage is a tiny recipe (function objects + pointers), not a copy of the data._
- On failure, Spark replays the lineage for only the lost partitions on other machines. — _Recovery is local: rebuild the few lost partitions, leave the rest untouched &mdash; no replicas needed._
- Note this requires transformations to be deterministic and the RDD to be immutable. — _Replaying the recipe must reproduce the exact same partition; mutation would break that guarantee._

**Answer:** Spark survives the failure through lineage: every RDD records the exact, deterministic chain of transformations that built it from durable input (a file in HDFS or cloud storage). When the worker dies, Spark replays that recipe for the lost partitions only on surviving machines and rebuilds them &mdash; so it never needed replicated copies; resilience comes from recomputation, not redundancy. The one requirement that makes it safe is determinism + immutability: because transformations always reproduce the same output and RDDs are never mutated in place, replaying the lineage gives byte-for-byte the same partition every time.

</details>